# DL-WAF — ModSecurity Baseline Evaluation
**MSc Thesis — Alex Ekwueme Federal University Ndufu-Alike (AE-FUNAI)**

Evaluates OWASP CRS v3 rule patterns (pure Python — no apt-get required)
against the CSIC 2010 test split and produces `modsecurity_metrics.json`.

### Before running:
1. Upload `csic_database.csv` as a Kaggle dataset and add it to this notebook
2. Update `CSV_PATH` in Cell 1 to match the path shown in the **Data** panel
3. Run All Cells (Run → Run All)

In [ ]:
# ── Cell 1: Imports and configuration ────────────────────────────────────────
import json, os, re, time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score,
    precision_score, recall_score,
)
from urllib.parse import urlparse, unquote_plus

CSV_PATH = "/kaggle/input/datasets/jesseolisaemaka/csic-database-local/csic_database.csv"
OUT_PATH = "/kaggle/working/modsecurity_metrics.json"

assert os.path.exists(CSV_PATH), f"CSV not found: {CSV_PATH}"
print(f"CSV found: {CSV_PATH}")

# Pre-filled from your local model/results/ JSON files
LSTM_METRICS = {
    "accuracy": 0.994978, "precision": 0.99679, "recall": 0.990957,
    "f1_score": 0.993865, "false_positive_rate": 0.002222,
    "confusion_matrix": {"tn": 5388, "fp": 12, "fn": 34, "tp": 3726}
}
CNN_METRICS = {
    "accuracy": 0.992467, "precision": 0.995969, "recall": 0.985638,
    "f1_score": 0.990777, "false_positive_rate": 0.002778,
    "confusion_matrix": {"tn": 5385, "fp": 15, "fn": 54, "tp": 3706}
}

print("Imports OK — ready to run Cell 2")

In [ ]:
# ── Cell 2: OWASP CRS v3 rules (pure Python) ─────────────────────────────────
F = re.IGNORECASE

CRS_RULES = [
    (
        "SQLi",
        re.compile(
            r"(?:(?:select|union|insert|update|delete|drop|alter|create|replace)\s+"
            r"(?:\*|all|distinct|from|into|table|database|schema|version))"
            r"|(?:--\s*$)|(?:;\s*(?:drop|delete|truncate|update))"
            r"|(?:0x[0-9a-f]+)|(?:char\s*\()|(?:exec\s*\()|(?:xp_cmdshell)"
            r"|(?:'\s*(?:or|and)\s*'?\s*(?:\d+|'[^']*')\s*=\s*(?:\d+|'[^']*'))"
            r"|(?:\bor\b\s+\d+\s*=\s*\d+)|(?:1=1)|(?:sleep\s*\(\s*\d)"
            r"|(?:benchmark\s*\()|(?:waitfor\s+delay)|(?:load_file\s*\()"
            r"|(?:into\s+(?:outfile|dumpfile))",
            F
        ),
        ["uri", "body"],
    ),
    (
        "XSS",
        re.compile(
            r"<\s*script[\s>]|<\s*img[^>]+onerror\s*=|javascript\s*:"
            r"|on(?:load|error|click|mouse\w+|key\w+|focus|blur|change|submit)\s*="
            r"|<\s*(?:iframe|object|embed|applet|form|svg|math)[\s>]"
            r"|expression\s*\(|vbscript\s*:|data\s*:\s*text/html"
            r"|document\.(?:cookie|write|location)|window\.location"
            r"|alert\s*\(|eval\s*\(",
            F
        ),
        ["uri", "body"],
    ),
    (
        "LFI",
        re.compile(
            r"\.\./|\.\.\\|\.\.\/"
            r"|(?:/etc/(?:passwd|shadow|hosts|group|issue|motd))"
            r"|(?:(?:boot|proc|sys)/)|(?:windows/system32)"
            r"|(?:cmd\.exe|/bin/(?:bash|sh|ksh|csh))",
            F
        ),
        ["uri", "body"],
    ),
    (
        "RFI",
        re.compile(
            r"(?:=\s*(?:https?|ftp|php|data)://)"
            r"|(?:include\s*\(?\s*['\"]?\s*(?:https?|ftp)://)",
            F
        ),
        ["uri", "body"],
    ),
    (
        "CMDi",
        re.compile(
            r"(?:[;&|`]\s*(?:ls|cat|wget|curl|chmod|chown|nc|ncat|nmap|ping|rm|mv|cp|echo|printf|id|whoami|uname|ifconfig|netstat)\b)"
            r"|(?:\$\([^)]+\))|(?:`[^`]+`)"
            r"|(?:\|\s*(?:nc|bash|sh|cmd)\b)",
            F
        ),
        ["uri", "body"],
    ),
    (
        "PHPi",
        re.compile(
            r"<\?php|\$(?:_GET|_POST|_COOKIE|_REQUEST|_SERVER|_FILES|_SESSION|GLOBALS)\s*\["
            r"|(?:base64_decode|passthru|shell_exec|system|phpinfo|popen|proc_open)\s*\(",
            F
        ),
        ["uri", "body"],
    ),
]


def crs_classify(method, url, body):
    """Return 1 (malicious) if any CRS rule fires, else 0."""
    targets = {"uri": unquote_plus(url), "body": unquote_plus(body)}
    for _, pattern, fields in CRS_RULES:
        for field in fields:
            if pattern.search(targets[field]):
                return 1
    return 0


print(f'{len(CRS_RULES)} CRS rule groups loaded (SQLi, XSS, LFI, RFI, CMDi, PHPi)')

In [ ]:
# ── Cell 3: Load dataset, reconstruct test split, evaluate ───────────────────
print('Loading dataset...')
df = pd.read_csv(CSV_PATH)
df['content'] = df['content'].fillna('')
print(f'  {len(df):,} rows loaded')

print('Reconstructing test split (70/15/15, random_state=42)...')
y_all = df['classification'].astype(int).values
_, df_tmp, _, y_tmp = train_test_split(
    df, y_all, test_size=0.30, stratify=y_all, random_state=42)
_, df_test, _, y_true_arr = train_test_split(
    df_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42)
df_test = df_test.reset_index(drop=True)
y_true  = y_true_arr.tolist()
print(f'  {len(df_test):,} test rows  |  malicious: {sum(y_true):,}  legitimate: {y_true.count(0):,}')

print(f'\nEvaluating {len(df_test):,} requests against CRS rules...')
y_pred    = []
latencies = []

for i, (_, row) in enumerate(df_test.iterrows(), 1):
    method = str(row.get('Method',  'GET') or 'GET').strip().upper()
    url    = str(row.get('URL',     '/')   or '/').strip()
    body   = str(row.get('content', '')   or '').strip()

    t0   = time.perf_counter()
    pred = crs_classify(method, url, body)
    latencies.append((time.perf_counter() - t0) * 1000)
    y_pred.append(pred)

    if i % 1000 == 0:
        print(f'  {i:,}/{len(df_test):,} done')

print('\nEvaluation complete')

In [ ]:
# ── Cell 4: Metrics, comparison table, save output ───────────────────────────
acc  = float(accuracy_score(y_true, y_pred))
prec = float(precision_score(y_true, y_pred, zero_division=0))
rec  = float(recall_score(y_true, y_pred, zero_division=0))
f1   = float(f1_score(y_true, y_pred, zero_division=0))
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel().tolist()
fpr  = round(fp / (fp + tn), 6) if (fp + tn) > 0 else 0.0

modsec_metrics = {
    "model":               "modsecurity_crs",
    "accuracy":            round(acc,  6),
    "precision":           round(prec, 6),
    "recall":              round(rec,  6),
    "f1_score":            round(f1,   6),
    "false_positive_rate": fpr,
    "confusion_matrix":    {"tn": tn, "fp": fp, "fn": fn, "tp": tp},
    "n_samples":           len(y_true),
    "mean_latency_ms":     round(float(np.mean(latencies)),           3),
    "p95_latency_ms":      round(float(np.percentile(latencies, 95)), 3),
}

with open(OUT_PATH, 'w') as f:
    json.dump(modsec_metrics, f, indent=2)
print(f'Saved -> {OUT_PATH}\n')

METRIC_ROWS = [
    ('Accuracy',            'accuracy',            True),
    ('Precision',           'precision',           True),
    ('Recall (TPR)',        'recall',              True),
    ('F1-Score',            'f1_score',            True),
    ('False Positive Rate', 'false_positive_rate', False),
]

print('=' * 72)
print('DL-WAF vs ModSecurity (OWASP CRS) — Chapter 5 Comparison')
print('=' * 72)
print(f"{'Metric':<22} {'LSTM':>10} {'CNN':>10} {'ModSec/CRS':>12} {'Best':>10}")
print('-' * 72)
for label, key, higher_better in METRIC_ROWS:
    v_lstm = LSTM_METRICS[key]
    v_cnn  = CNN_METRICS[key]
    v_ms   = modsec_metrics[key]
    scores = {'LSTM': v_lstm, 'CNN': v_cnn, 'ModSec': v_ms}
    best   = max(scores, key=lambda k: scores[k] if higher_better else -scores[k])
    print(f"{label:<22} {v_lstm*100:>9.4f}% {v_cnn*100:>9.4f}% {v_ms*100:>11.4f}%  <- {best}")
print('-' * 72)

print(f'\nConfusion matrix (ModSec/CRS):')
print(f'  TP={tp:,}  FP={fp:,}  FN={fn:,}  TN={tn:,}')
print(f"\nLatency: mean={modsec_metrics['mean_latency_ms']} ms  p95={modsec_metrics['p95_latency_ms']} ms")
print('\n-> Download modsecurity_metrics.json from the Output tab')
print('   Copy it to your local model/results/ folder.')